# Setup

In [0]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))

from src.config import *
from src.params import resolve_params
from src.pipelines.raw_to_bronze import run_raw_to_bronze
from src.pipelines.reference import load_categories
from src.pipelines.bronze_to_silver import run_bronze_to_silver

def run_pipeline(load_type, start, end, regions="", batch_id=""):
    """Raw -> Bronze -> Silver for any date range. Parameters only, no code edits."""
    s, e, r, b = resolve_params(load_type, start, end, regions, batch_id)
    print("BRONZE:", run_raw_to_bronze(spark, ["api"], s, e, r, load_type, b))
    load_categories(spark, load_type, b)
    run_bronze_to_silver(spark, s, e, load_type, b)
    print("SILVER done | batch_id =", b)
    return b

def fingerprint(table, row_expr, where="1=1"):
    """Row count + a hash of the table's content + latest load_timestamp."""
    return spark.sql(f"""
        SELECT COUNT(*) AS row_count,
               md5(concat_ws('', sort_array(collect_list({row_expr})))) AS content_hash,
               MAX(load_timestamp) AS max_load_ts
        FROM {table} WHERE {where}""").first()

SILVER_ROW = "concat(video_id, region_code, CAST(snapshot_date AS STRING), row_hash)"
BRONZE_ROW = "bronze_record_id"

def show_logs(batch_id):
    display(spark.table(T_RUN_LOGS).filter(f"batch_id = '{batch_id}'")
        .select("layer", "load_type", "parameter", "status", "rows_read",
                "rows_inserted", "rows_updated", "rows_quarantined", "start_time", "end_time")
        .orderBy("start_time"))

# 1. Explicit Schema

In [0]:
from src.schemas import API_ITEM_SCHEMA, KAGGLE_CSV_SCHEMA
print("READ schema used before reading raw files (no inferSchema):")
print(API_ITEM_SCHEMA.simpleString())
print()
for t in (T_BRONZE_API, T_BRONZE_KAGGLE, T_SILVER_VIDEO, T_SILVER_CAT, T_QUARANTINE, T_RUN_LOGS):
    print("=====", t); spark.table(t).printSchema()

READ schema used before reading raw files (no inferSchema):
struct<kind:string,etag:string,id:string,snippet:struct<publishedAt:string,channelId:string,title:string,description:string,channelTitle:string,tags:array<string>,categoryId:string>,contentDetails:struct<duration:string>,statistics:struct<viewCount:string,likeCount:string,commentCount:string>>

===== workspace.youtube_trending.bronze_api_videos
root
 |-- raw_json: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- kind: string (nullable = true)
 |    |-- etag: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- snippet: struct (nullable = true)
 |    |    |-- publishedAt: string (nullable = true)
 |    |    |-- channelId: string (nullable = true)
 |    |    |-- title: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- channelTitle: string (nullable = true)
 |    |    |-- tags: array (nullable = true)
 |    |    |    |-- element: string (cont

# 2. Idempotency

In [0]:
W_B = "ingestion_date BETWEEN '2026-09-26' AND '2026-09-27'"
W_S = "snapshot_date BETWEEN '2026-09-26' AND '2026-09-27'"
before_b = fingerprint(T_BRONZE_API, BRONZE_ROW, W_B)
before_s = fingerprint(T_SILVER_VIDEO, SILVER_ROW, W_S)
print("BEFORE bronze:", before_b); print("BEFORE silver:", before_s)

BEFORE bronze: Row(row_count=996, content_hash='6865c7cb702ca0309229c41f7560efb7', max_load_ts=datetime.datetime(2026, 10, 11, 16, 53, 4, 356890))
BEFORE silver: Row(row_count=996, content_hash='2b14ab9dbbc2b3045d42c8b5d43a1c29', max_load_ts=datetime.datetime(2026, 10, 11, 16, 55, 42, 566776))


In [0]:
bid = run_pipeline("incremental", "2026-09-26", "2026-09-27", "", "idem_rerun")

BRONZE: {'succeeded': 20, 'failed': []}
SILVER done | batch_id = idem_rerun


In [0]:
after_b = fingerprint(T_BRONZE_API, BRONZE_ROW, W_B)
after_s = fingerprint(T_SILVER_VIDEO, SILVER_ROW, W_S)
print("AFTER  bronze:", after_b); print("AFTER  silver:", after_s)
print("Bronze identical:", before_b == after_b)
print("Silver identical:", before_s == after_s)

display(spark.sql(f"""
  SELECT COUNT(*) AS rows, COUNT(DISTINCT video_id, region_code, snapshot_date) AS distinct_keys
  FROM {T_SILVER_VIDEO}"""))
show_logs("idem_rerun")

AFTER  bronze: Row(row_count=996, content_hash='6865c7cb702ca0309229c41f7560efb7', max_load_ts=datetime.datetime(2026, 10, 11, 16, 53, 4, 356890))
AFTER  silver: Row(row_count=996, content_hash='2b14ab9dbbc2b3045d42c8b5d43a1c29', max_load_ts=datetime.datetime(2026, 10, 11, 16, 55, 42, 566776))
Bronze identical: True
Silver identical: True


rows,distinct_keys
1495,1495


layer,load_type,parameter,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=PK/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:43:01.429Z,2026-10-11T16:43:20.050Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=US/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:43:22.575Z,2026-10-11T16:43:36.845Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=IN/videos.jsonl,SUCCESS,49,0,0,0,2026-10-11T16:43:38.777Z,2026-10-11T16:43:52.511Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=GB/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:43:54.827Z,2026-10-11T16:44:09.251Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=CA/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:44:11.238Z,2026-10-11T16:44:26.101Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=AU/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:44:28.084Z,2026-10-11T16:44:41.115Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=DE/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:44:43.173Z,2026-10-11T16:44:55.965Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=FR/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:44:57.848Z,2026-10-11T16:45:11.446Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=JP/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:45:13.642Z,2026-10-11T16:45:27.654Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=BR/videos.jsonl,SUCCESS,50,0,0,0,2026-10-11T16:45:29.492Z,2026-10-11T16:45:45.352Z


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {T_SILVER_VIDEO}")
        .filter("operation = 'MERGE'")
        .select("version", "timestamp", "operationMetrics").limit(3))

version,timestamp,operationMetrics
10,2026-10-11T18:12:02.000Z,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3170, materializeSourceTimeMs -> 750, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1125, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 996, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1269)"
9,2026-10-11T17:50:46.000Z,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3524, materializeSourceTimeMs -> 903, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1298, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 497, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1284)"
8,2026-10-11T17:40:24.000Z,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 325453, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5703, materializeSourceTimeMs -> 1901, numTargetRowsInserted -> 497, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2363, numTargetRowsUpdated -> 0, numOutputRows -> 497, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 497, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1380)"


# 3. Backfill

In [0]:
ref_s = fingerprint(T_SILVER_VIDEO, SILVER_ROW, "snapshot_date = '2026-09-26'")
ref_b = fingerprint(T_BRONZE_API, BRONZE_ROW, "ingestion_date = '2026-09-26'")
print("reference silver 09-26:", ref_s); print("reference bronze 09-26:", ref_b)

spark.sql(f"DELETE FROM {T_SILVER_VIDEO} WHERE snapshot_date = '2026-09-26'")
spark.sql(f"DELETE FROM {T_BRONZE_API} WHERE ingestion_date = '2026-09-26'")

for name, t, c in [("bronze", T_BRONZE_API, "ingestion_date"), ("silver", T_SILVER_VIDEO, "snapshot_date")]:
    print(name, "after delete:")
    display(spark.sql(f"SELECT {c} AS d, COUNT(*) AS rows FROM {t} GROUP BY {c} ORDER BY {c}"))

reference silver 09-26: Row(row_count=499, content_hash='d56d997fb86c781476765d088843dc56', max_load_ts=datetime.datetime(2026, 10, 11, 16, 55, 42, 566776))
reference bronze 09-26: Row(row_count=499, content_hash='c5d33318438a5e61acce94b292e9dedf', max_load_ts=datetime.datetime(2026, 10, 11, 16, 53, 4, 356890))
bronze after delete:


d,rows
2026-09-27,497
2026-09-28,4
2026-10-11,497


silver after delete:


d,rows
2026-09-27,497
2026-09-28,2
2026-10-11,497


In [0]:
bid = run_pipeline("backfill", "2026-09-26", "2026-09-27", "", "backfill_demo")

BRONZE: {'succeeded': 20, 'failed': []}
SILVER done | batch_id = backfill_demo


In [0]:
new_s = fingerprint(T_SILVER_VIDEO, SILVER_ROW, "snapshot_date = '2026-09-26'")
new_b = fingerprint(T_BRONZE_API, BRONZE_ROW, "ingestion_date = '2026-09-26'")
print("rows restored  | bronze:", new_b.row_count == ref_b.row_count, "| silver:", new_s.row_count == ref_s.row_count)
print("content identical | bronze:", new_b.content_hash == ref_b.content_hash,
      "| silver:", new_s.content_hash == ref_s.content_hash)
show_logs("backfill_demo")

rows restored  | bronze: True | silver: True
content identical | bronze: True | silver: True


layer,load_type,parameter,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=PK/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:50:37.651Z,2026-10-11T16:50:51.480Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=US/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:50:53.069Z,2026-10-11T16:51:06.741Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=IN/videos.jsonl,SUCCESS,49,49,0,0,2026-10-11T16:51:08.360Z,2026-10-11T16:51:21.243Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=GB/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:51:22.797Z,2026-10-11T16:51:35.037Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=CA/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:51:36.637Z,2026-10-11T16:51:48.801Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=AU/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:51:50.671Z,2026-10-11T16:52:05.378Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=DE/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:52:07.211Z,2026-10-11T16:52:20.360Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=FR/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:52:22.148Z,2026-10-11T16:52:35.649Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=JP/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:52:37.375Z,2026-10-11T16:52:53.264Z
raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-26/region=BR/videos.jsonl,SUCCESS,50,50,0,0,2026-10-11T16:52:55.208Z,2026-10-11T16:53:08.695Z


#4. Schema Drift

In [0]:
spark.sql(f"DELETE FROM {T_BRONZE_API} WHERE ingestion_date = '2026-09-28'")
spark.sql(f"DELETE FROM {T_SILVER_VIDEO} WHERE snapshot_date = '2026-09-28'")
for t in (T_QUARANTINE, T_DRIFT_LOGS):
    try: spark.sql(f"DELETE FROM {t} WHERE batch_id = 'drift_demo'")
    except Exception: pass          # tables may not exist on the very first run

In [0]:
d = f"{RAW_ROOT}/youtube_api/ingestion_date=2026-09-28/region=PK"
os.makedirs(d, exist_ok=True)

def item(vid, stats):
    return {"kind": "youtube#video", "id": vid,
            "snippet": {"publishedAt": "2026-09-27T10:00:00Z", "channelId": "UCdemo",
                        "title": f"Drift test {vid}", "categoryId": "10"},
            "contentDetails": {"duration": "PT3M5S"}, "statistics": stats}

lines = [
    json.dumps(item("drift001", {"viewCount": "100", "likeCount": "5", "commentCount": "1", "shareCount": "9"})),  # new column
    json.dumps(item("drift002", {"viewCount": "N/A", "likeCount": "5", "commentCount": "1"})),                      # bad type
    json.dumps(item("drift003", {"viewCount": 250, "likeCount": 7, "commentCount": 2})),                            # numbers, not strings
    "{this is not valid json",                                                                                       # corrupt
]
open(f"{d}/videos.jsonl", "w", encoding="utf-8").write("\n".join(lines))
print("written:", f"{d}/videos.jsonl")

written: /Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-28/region=PK/videos.jsonl


In [0]:
bid = run_pipeline("incremental", "2026-09-28", "2026-09-28", "PK", "drift_demo")

BRONZE: {'succeeded': 1, 'failed': []}
SILVER done | batch_id = drift_demo


In [0]:
print("1) New column detected and logged:")
display(spark.table(T_DRIFT_LOGS).filter("batch_id = 'drift_demo'")
        .select("source_file", "scope", "new_columns", "load_timestamp"))

print("2) Bronze keeps everything, including the corrupt line:")
display(spark.table(T_BRONZE_API).filter("ingestion_date = '2026-09-28'")
        .select("is_parseable", "payload.id", "raw_json"))

print("3) Silver holds only the valid rows:")
display(spark.table(T_SILVER_VIDEO).filter("snapshot_date = '2026-09-28'")
        .select("video_id", "view_count", "like_count", "comment_count", "batch_id"))

print("4) Quarantine holds the bad records with reasons:")
display(spark.table(T_QUARANTINE).filter("batch_id = 'drift_demo'")
        .select("source_table", "reason", "raw_record"))

show_logs("drift_demo")

1) New column detected and logged:


source_file,scope,new_columns,load_timestamp
/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-28/region=PK/videos.jsonl,statistics,shareCount,2026-10-11T18:17:45.762Z


2) Bronze keeps everything, including the corrupt line:


is_parseable,id,raw_json
false,null,{this is not valid json
true,drift002,"{""kind"": ""youtube#video"", ""id"": ""drift002"", ""snippet"": {""publishedAt"": ""2026-09-27T10:00:00Z"", ""channelId"": ""UCdemo"", ""title"": ""Drift test drift002"", ""categoryId"": ""10""}, ""contentDetails"": {""duration"": ""PT3M5S""}, ""statistics"": {""viewCount"": ""N/A"", ""likeCount"": ""5"", ""commentCount"": ""1""}}"
true,drift003,"{""kind"": ""youtube#video"", ""id"": ""drift003"", ""snippet"": {""publishedAt"": ""2026-09-27T10:00:00Z"", ""channelId"": ""UCdemo"", ""title"": ""Drift test drift003"", ""categoryId"": ""10""}, ""contentDetails"": {""duration"": ""PT3M5S""}, ""statistics"": {""viewCount"": 250, ""likeCount"": 7, ""commentCount"": 2}}"
true,drift001,"{""kind"": ""youtube#video"", ""id"": ""drift001"", ""snippet"": {""publishedAt"": ""2026-09-27T10:00:00Z"", ""channelId"": ""UCdemo"", ""title"": ""Drift test drift001"", ""categoryId"": ""10""}, ""contentDetails"": {""duration"": ""PT3M5S""}, ""statistics"": {""viewCount"": ""100"", ""likeCount"": ""5"", ""commentCount"": ""1"", ""shareCount"": ""9""}}"


3) Silver holds only the valid rows:


video_id,view_count,like_count,comment_count,batch_id
drift001,100,5,1,drift_demo
drift003,250,7,2,drift_demo


4) Quarantine holds the bad records with reasons:


source_table,reason,raw_record
youtube_api,cast_failed:view_count,"{""kind"": ""youtube#video"", ""id"": ""drift002"", ""snippet"": {""publishedAt"": ""2026-09-27T10:00:00Z"", ""channelId"": ""UCdemo"", ""title"": ""Drift test drift002"", ""categoryId"": ""10""}, ""contentDetails"": {""duration"": ""PT3M5S""}, ""statistics"": {""viewCount"": ""N/A"", ""likeCount"": ""5"", ""commentCount"": ""1""}}"
youtube_api,unparseable_json_or_missing_id,{this is not valid json


layer,load_type,parameter,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-28/region=PK/videos.jsonl,SUCCESS,4,4,0,0,2026-10-11T17:00:40.461Z,2026-10-11T17:00:51.966Z
reference_to_silver,incremental,/Volumes/workspace/youtube_trending/raw/reference/categories.csv,SUCCESS,18,0,0,0,2026-10-11T17:00:54.352Z,2026-10-11T17:01:00.319Z
bronze_to_silver,incremental,ingestion_date 2026-09-28..2026-09-28,SUCCESS,4,2,0,2,2026-10-11T17:01:02.886Z,2026-10-11T17:01:14.536Z
raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-28/region=PK/videos.jsonl,SUCCESS,4,4,0,0,2026-10-11T18:17:42.758Z,2026-10-11T18:17:52.390Z
reference_to_silver,incremental,/Volumes/workspace/youtube_trending/raw/reference/categories.csv,SUCCESS,32,0,0,0,2026-10-11T18:17:54.854Z,2026-10-11T18:18:00.935Z
bronze_to_silver,incremental,ingestion_date 2026-09-28..2026-09-28,SUCCESS,4,2,0,2,2026-10-11T18:18:03.326Z,2026-10-11T18:18:17.367Z


#5. Final Log Table

In [0]:
display(spark.table(T_RUN_LOGS)
        .select("batch_id", "layer", "load_type", "parameter", "status",
                "start_time", "end_time", "rows_read", "rows_inserted", "rows_updated", "rows_quarantined")
        .orderBy("start_time", ascending=False))

batch_id,layer,load_type,parameter,status,start_time,end_time,rows_read,rows_inserted,rows_updated,rows_quarantined
drift_demo,bronze_to_silver,incremental,ingestion_date 2026-09-28..2026-09-28,SUCCESS,2026-10-11T18:18:03.326Z,2026-10-11T18:18:17.367Z,4,2,0,2
drift_demo,reference_to_silver,incremental,/Volumes/workspace/youtube_trending/raw/reference/categories.csv,SUCCESS,2026-10-11T18:17:54.854Z,2026-10-11T18:18:00.935Z,32,0,0,0
drift_demo,raw_to_bronze,incremental,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-28/region=PK/videos.jsonl,SUCCESS,2026-10-11T18:17:42.758Z,2026-10-11T18:17:52.390Z,4,4,0,0
backfill_demo,bronze_to_silver,backfill,ingestion_date 2026-09-26..2026-09-27,SUCCESS,2026-10-11T18:17:17.367Z,2026-10-11T18:17:29.657Z,996,499,0,0
backfill_demo,reference_to_silver,backfill,/Volumes/workspace/youtube_trending/raw/reference/categories.csv,SUCCESS,2026-10-11T18:17:08.605Z,2026-10-11T18:17:15.035Z,32,0,0,0
backfill_demo,raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-27/region=BR/videos.jsonl,SUCCESS,2026-10-11T18:16:55.615Z,2026-10-11T18:17:06.648Z,50,0,0,0
backfill_demo,raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-27/region=JP/videos.jsonl,SUCCESS,2026-10-11T18:16:41.974Z,2026-10-11T18:16:54.026Z,50,0,0,0
backfill_demo,raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-27/region=FR/videos.jsonl,SUCCESS,2026-10-11T18:16:28.351Z,2026-10-11T18:16:40.280Z,50,0,0,0
backfill_demo,raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-27/region=DE/videos.jsonl,SUCCESS,2026-10-11T18:16:15.188Z,2026-10-11T18:16:26.755Z,50,0,0,0
backfill_demo,raw_to_bronze,backfill,/Volumes/workspace/youtube_trending/raw/youtube_api/ingestion_date=2026-09-27/region=AU/videos.jsonl,SUCCESS,2026-10-11T18:16:01.830Z,2026-10-11T18:16:13.625Z,50,0,0,0
